# Nettoyade de la data

In [17]:
import numpy as np
import pandas as pd

Chargement de la dataset

In [18]:
df = pd.read_csv("../data/raw/transactions_bancaires_synthetiques_brutes.csv")
df.head()

,transaction_id,customer_id,transaction_date,transaction_time,account_type,payment_method,direction,amount,balance_after,narration,merchant_name,merchant_city,merchant_country
0,TX0000001,CUST0001,2024-01-01,09:34,Courant,Virement instantané,Credit,11998.10,19846.02,virement employeur,Salaire,Rabat,MA
1,TX0000003,CUST0001,2024-01-02,09:40,Courant,Wallet,Debit,21.55,19824.47,TRANSPORT CAREEM Rabat,Careem,Rabat,MA
2,TX0000002,CUST0001,2024-01-02,11:16,Courant,Carte,Debit,147.03,19677.44,CARBURANT SHELL Rabat,Shell,Rabat,MA
3,TX0000004,CUST0001,2024-01-03,08:53,Courant,Carte,Debit,403.15,19274.29,JUMIA MAROC,Jumia,Rabat,MA
4,TX0000005,CUST0001,2024-01-03,18:49,Courant,Virement,Debit,775.01,18499.28,LOYER APPARTEMENT Rabat,Loyer Appartement,Rabat,MA


dimensions et colonnes

In [19]:
print(df.shape)
print(df.columns.tolist())

(46702, 13)
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'payment_method', 'direction', 'amount', 'balance_after', 'narration', 'merchant_name', 'merchant_city', 'merchant_country']


types des colonnes

In [20]:
print(df.dtypes)

transaction_id       object
customer_id          object
transaction_date     object
transaction_time     object
account_type         object
payment_method       object
direction            object
amount              float64
balance_after       float64
narration            object
merchant_name        object
merchant_city        object
merchant_country     object
dtype: object


Conversion de la date

In [21]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df[["transaction_date"]].head()

,transaction_date
0,2024-01-01
1,2024-01-02
2,2024-01-02
3,2024-01-03
4,2024-01-03


Conversion de l'heure

In [22]:
df["transaction_time"] = pd.to_datetime(
    df["transaction_time"],
    format="%H:%M", 
    errors="coerce").dt.time
df[["transaction_time"]].head()


,transaction_time
0,09:34:00
1,09:40:00
2,11:16:00
3,08:53:00
4,18:49:00


Création du datetime complet

In [23]:
df["transaction_datetime"] = pd.to_datetime(df["transaction_date"].astype(str) + " " + df["transaction_time"].astype(str), errors='coerce')

df[["transaction_date", "transaction_time", "transaction_datetime"]].head()

,transaction_date,transaction_time,transaction_datetime
0,2024-01-01,09:34:00,2024-01-01 09:34:00
1,2024-01-02,09:40:00,2024-01-02 09:40:00
2,2024-01-02,11:16:00,2024-01-02 11:16:00
3,2024-01-03,08:53:00,2024-01-03 08:53:00
4,2024-01-03,18:49:00,2024-01-03 18:49:00


Création des variables temporelles

In [24]:
df["year"] = df["transaction_datetime"].dt.year
df["month"] = df["transaction_datetime"].dt.month
df["day"] = df["transaction_datetime"].dt.day
df["hour"] = df["transaction_datetime"].dt.hour
df["weekday"] = df["transaction_datetime"].dt.day_name()
df["month_name"] = df["transaction_datetime"].dt.month_name()

df[["year", "month", "day", "hour", "weekday", "month_name"]].head()

,year,month,day,hour,weekday,month_name
0,2024,1,1,9,Monday,January
1,2024,1,2,9,Tuesday,January
2,2024,1,2,11,Tuesday,January
3,2024,1,3,8,Wednesday,January
4,2024,1,3,18,Wednesday,January


Standardisation des textes

In [25]:
text_cols = [
    "account_type",
    "payment_method",
    "direction",
    "narration",
    "merchant_name",
    "merchant_city",
    "merchant_country"
]

for col in text_cols : 
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)
    
df[text_cols].head()

,account_type,payment_method,direction,narration,merchant_name,merchant_city,merchant_country
0,Courant,Virement instantané,Credit,virement employeur,Salaire,Rabat,MA
1,Courant,Wallet,Debit,TRANSPORT CAREEM Rabat,Careem,Rabat,MA
2,Courant,Carte,Debit,CARBURANT SHELL Rabat,Shell,Rabat,MA
3,Courant,Carte,Debit,JUMIA MAROC,Jumia,Rabat,MA
4,Courant,Virement,Debit,LOYER APPARTEMENT Rabat,Loyer Appartement,Rabat,MA


Narration nettoyé

In [26]:
df["clean_narration"] = (
    df["narration"]
    .str.upper()
    .str.strip()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

df[["narration", "clean_narration"]].head(10)

,narration,clean_narration
0,virement employeur,VIREMENT EMPLOYEUR
1,TRANSPORT CAREEM Rabat,TRANSPORT CAREEM RABAT
2,CARBURANT SHELL Rabat,CARBURANT SHELL RABAT
3,JUMIA MAROC,JUMIA MAROC
4,LOYER APPARTEMENT Rabat,LOYER APPARTEMENT RABAT
5,RETRAIT GAB Tanger,RETRAIT GAB TANGER
6,VIR LOYER Fès,VIR LOYER FÈS
7,PAIEMENT CARREFOUR Rabat,PAIEMENT CARREFOUR RABAT
8,MEGARAMA Rabat,MEGARAMA RABAT
9,ORANGE MOBILE,ORANGE MOBILE


Nom marchand nettoyé

In [27]:
df["clean_merchant_name"] = (
    df["merchant_name"]
    .str.upper()
    .str.strip()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

df[["merchant_name", "clean_merchant_name"]].head(10)

,merchant_name,clean_merchant_name
0,Salaire,SALAIRE
1,Careem,CAREEM
2,Shell,SHELL
3,Jumia,JUMIA
4,Loyer Appartement,LOYER APPARTEMENT
5,ATM,ATM
6,Loyer Appartement,LOYER APPARTEMENT
7,Carrefour,CARREFOUR
8,Cinéma Megarama,CINÉMA MEGARAMA
9,Orange,ORANGE


Colonnes débit / crédit

In [28]:
df["is_debit"] = np.where(df["direction"].str.lower() == "debit", 1, 0)
df["is_credit"] = np.where(df["direction"].str.lower() == "credit", 1, 0)

df[["direction", "is_debit", "is_credit"]].head(10)

,direction,is_debit,is_credit
0,Credit,0,1
1,Debit,1,0
2,Debit,1,0
3,Debit,1,0
4,Debit,1,0
5,Debit,1,0
6,Debit,1,0
7,Debit,1,0
8,Debit,1,0
9,Debit,1,0


Type métier de transaction

In [29]:
df["transaction_type"] = np.where(df["is_debit"] == 1, "Dépense", "Revenu")
df[["direction", "transaction_type"]].head(10)

,direction,transaction_type
0,Credit,Revenu
1,Debit,Dépense
2,Debit,Dépense
3,Debit,Dépense
4,Debit,Dépense
5,Debit,Dépense
6,Debit,Dépense
7,Debit,Dépense
8,Debit,Dépense
9,Debit,Dépense


Période de la journée

In [30]:
def get_time_period(hour):
    if pd.isna(hour):
        return "Inconnu"
    elif 6 <= hour < 12:
        return "Matin"
    elif 12 <= hour < 18:
        return "Après-midi"
    elif 18 <= hour < 22:
        return "Soir"
    else:
        return "Nuit"

df["time_period"] = df["hour"].apply(get_time_period)

df[["hour", "time_period"]].head(10)

,hour,time_period
0,9,Matin
1,9,Matin
2,11,Matin
3,8,Matin
4,18,Soir
5,7,Matin
6,18,Soir
7,20,Soir
8,15,Après-midi
9,17,Après-midi


Vérification finale

In [31]:
print(df.dtypes)
print(df.isna().sum()[df.isna().sum() > 0])

df[[
    "transaction_date", "transaction_time", "transaction_datetime",
    "year", "month", "day", "hour", "weekday", "month_name",
    "clean_narration", "clean_merchant_name",
    "is_debit", "is_credit", "transaction_type", "time_period"
]].head(10)

transaction_id                  object
customer_id                     object
transaction_date        datetime64[ns]
transaction_time                object
account_type                    object
payment_method                  object
direction                       object
amount                         float64
balance_after                  float64
narration                       object
merchant_name                   object
merchant_city                   object
merchant_country                object
transaction_datetime    datetime64[ns]
year                             int32
month                            int32
day                              int32
hour                             int32
weekday                         object
month_name                      object
clean_narration                 object
clean_merchant_name             object
is_debit                         int64
is_credit                        int64
transaction_type                object
time_period              

,transaction_date,transaction_time,transaction_datetime,year,month,day,hour,weekday,month_name,clean_narration,clean_merchant_name,is_debit,is_credit,transaction_type,time_period
0,2024-01-01,09:34:00,2024-01-01 09:34:00,2024,1,1,9,Monday,January,VIREMENT EMPLOYEUR,SALAIRE,0,1,Revenu,Matin
1,2024-01-02,09:40:00,2024-01-02 09:40:00,2024,1,2,9,Tuesday,January,TRANSPORT CAREEM RABAT,CAREEM,1,0,Dépense,Matin
2,2024-01-02,11:16:00,2024-01-02 11:16:00,2024,1,2,11,Tuesday,January,CARBURANT SHELL RABAT,SHELL,1,0,Dépense,Matin
3,2024-01-03,08:53:00,2024-01-03 08:53:00,2024,1,3,8,Wednesday,January,JUMIA MAROC,JUMIA,1,0,Dépense,Matin
4,2024-01-03,18:49:00,2024-01-03 18:49:00,2024,1,3,18,Wednesday,January,LOYER APPARTEMENT RABAT,LOYER APPARTEMENT,1,0,Dépense,Soir
5,2024-01-04,07:07:00,2024-01-04 07:07:00,2024,1,4,7,Thursday,January,RETRAIT GAB TANGER,ATM,1,0,Dépense,Matin
6,2024-01-04,18:19:00,2024-01-04 18:19:00,2024,1,4,18,Thursday,January,VIR LOYER FÈS,LOYER APPARTEMENT,1,0,Dépense,Soir
7,2024-01-04,20:51:00,2024-01-04 20:51:00,2024,1,4,20,Thursday,January,PAIEMENT CARREFOUR RABAT,CARREFOUR,1,0,Dépense,Soir
8,2024-01-05,15:02:00,2024-01-05 15:02:00,2024,1,5,15,Friday,January,MEGARAMA RABAT,CINÉMA MEGARAMA,1,0,Dépense,Après-midi
9,2024-01-05,17:42:00,2024-01-05 17:42:00,2024,1,5,17,Friday,January,ORANGE MOBILE,ORANGE,1,0,Dépense,Après-midi


Sauvegarde

In [32]:
df.to_csv("../data/processed/transactions_nettoyees.csv", index=False)
print("Fichier sauvegardé : transactions_nettoyees.csv")

Fichier sauvegardé : transactions_nettoyees.csv
